# Demonstrasi: Transformasi & Representasi Data — PIF304 Pengantar Data Sains
**Pertemuan 6 — Sub-CPMK S03: Memilih transformasi dan representasi data**

Data yang sudah *bersih* belum tentu *siap dianalisis*. Hari ini kita mengubah **bentuk** dan **representasi** data tanpa mengubah **maknanya**:

1. Tidy data (bentuk lebar ↔ panjang)
2. Encoding: mengubah kategori menjadi angka
3. Scaling: menyamakan skala angka
4. Agregasi: meringkas per kelompok
5. Penggabungan data: `merge` dan `concat`
6. Pengantar feature engineering
7. Keterlacakan: dataset analisis + catatan transformasi

**Dataset:** `Data_Survei_Bersih_PDS.xlsx` — survei 45 responden dengan kolom tambahan `jarak_km_bersih` dan `jarak_dikoreksi` (hasil keputusan Pertemuan 5).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
sns.set_theme(style="whitegrid")
# Palet grafik (sudah diuji aman untuk buta warna): satu seri = TEAL; pembanding = ORANGE, VIOLET; latar = GREY
NAVY, TEAL, ORANGE, VIOLET, GREY = "#13233F", "#1293A6", "#E4683A", "#6C63C9", "#B8BDC4"
PALET = [TEAL, ORANGE, VIOLET]

### Unggah File Data
Jalankan sel di bawah, klik **Choose Files**, lalu pilih **`Data_Survei_Bersih_PDS.xlsx`** yang sudah Anda unduh dari LMS.

In [ ]:
try:
    from google.colab import files
    print("Silakan pilih file: Data_Survei_Bersih_PDS.xlsx")
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise ValueError("Belum ada file yang diunggah. Jalankan ulang sel ini dan pilih file-nya.")
    nama_file = list(uploaded.keys())[0]
except ImportError:
    # Di luar Google Colab (mis. Jupyter lokal): letakkan file di folder yang sama dengan notebook
    nama_file = "Data_Survei_Bersih_PDS.xlsx"

df = pd.read_excel(nama_file)   # sheet pertama = "Data"
print("File:", nama_file, "| ukuran:", df.shape)
df.head()

In [ ]:
n_awal = len(df)
transformasi = []           # catatan transformasi (keterlacakan)
def catat(nama, masukan, keluaran, alasan):
    transformasi.append({"transformasi": nama, "kolom_masukan": masukan, "kolom_keluaran": keluaran, "alasan": alasan})
print("Jumlah baris awal:", n_awal)

## 1. Tidy Data
Tiga aturan *tidy data* (Wickham, 2014):
1. Setiap **variabel** adalah satu kolom.
2. Setiap **observasi** adalah satu baris.
3. Setiap **nilai** adalah satu sel.

Data survei kita sudah tidy (1 baris = 1 responden). Tetapi laporan sering datang dalam bentuk **lebar**, misalnya rekap berikut:

In [ ]:
lebar = pd.crosstab(df["semester"], df["waktu_belajar_favorit"]).reset_index()
lebar.columns.name = None
lebar

Tabel ini enak dibaca manusia, tetapi **tidak tidy**: "Malam", "Pagi", "Siang" adalah *nilai* dari variabel `waktu_belajar`, bukan variabel. `melt` mengubahnya menjadi bentuk **panjang** yang mudah difilter, dikelompokkan, dan digambar.

In [ ]:
panjang = lebar.melt(id_vars="semester", var_name="waktu_belajar", value_name="jumlah")
panjang

In [ ]:
# Kebalikannya: panjang -> lebar dengan pivot
panjang.pivot(index="semester", columns="waktu_belajar", values="jumlah")

## 2. Encoding
Banyak metode analisis hanya menerima angka. Cara mengubahnya **bergantung pada skala ukur**.

### 2a. Ordinal → kode berurutan

In [ ]:
peta_ai = {"Kadang-kadang": 1, "Sering": 2, "Sangat sering": 3}
df["frekuensi_pakai_ai_kode"] = df["frekuensi_pakai_ai"].map(peta_ai)
catat("encoding ordinal", "frekuensi_pakai_ai", "frekuensi_pakai_ai_kode", "Kategori berurutan; peta eksplisit 1-3")
df[["frekuensi_pakai_ai", "frekuensi_pakai_ai_kode"]].drop_duplicates().sort_values("frekuensi_pakai_ai_kode")

Kode 1–2–3 menjaga **urutan**, tetapi mengandaikan jarak antarkategori sama. Itu asumsi; tuliskan di catatan.

### 2b. Nominal → one-hot
Untuk kategori **tanpa urutan**, memberi kode 1–2–3 akan menciptakan urutan palsu ("Sepeda motor" > "Jalan kaki"?). Gunakan *one-hot encoding*: satu kolom 0/1 per kategori.

In [ ]:
onehot = pd.get_dummies(df["moda_transportasi"], prefix="moda", dtype=int)
df = pd.concat([df, onehot], axis=1)
catat("one-hot", "moda_transportasi", ", ".join(onehot.columns), "Kategori nominal; tanpa urutan")
df[["moda_transportasi"] + list(onehot.columns)].drop_duplicates()

**Perhatikan:** kategori dengan anggota sangat sedikit (Ojek online = 1 orang) menghasilkan kolom yang hampir seluruhnya nol, dan sekaligus membuat orang itu mudah dikenali. Menggabungkan kategori kecil menjadi "Lainnya" sering lebih bijak — tetapi itu juga keputusan yang harus dicatat.

## 3. Scaling
`usia` berkisar 18–30, `waktu_tempuh_menit` 2–25, `jarak_km_bersih` 0–10. Metode yang memakai jarak antartitik akan didominasi kolom berangka besar. Dua cara umum:

- **Min-max:** x' = (x − min) / (max − min) → rentang 0–1
- **Standardisasi (z-score):** z = (x − mean) / simpangan baku → mean 0, simpangan baku 1

In [ ]:
kolom = ["usia", "waktu_tempuh_menit", "jarak_km_bersih"]
minmax = (df[kolom] - df[kolom].min()) / (df[kolom].max() - df[kolom].min())
zscore = (df[kolom] - df[kolom].mean()) / df[kolom].std()

pd.concat({"asli": df[kolom].describe().loc[["min", "max", "mean", "std"]],
           "min-max": minmax.describe().loc[["min", "max", "mean", "std"]],
           "z-score": zscore.describe().loc[["min", "max", "mean", "std"]]}, axis=1).round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, (judul, d) in zip(axes, {"Asli": df[kolom], "Min-max": minmax, "Z-score": zscore}.items()):
    sns.boxplot(data=d, orient="h", ax=ax, palette=PALET); ax.set_title(judul)
plt.tight_layout(); plt.show()

**Catatan:** scaling mengubah **angka**, bukan **bentuk** distribusi atau urutan data. Min-max peka terhadap nilai ekstrem (bayangkan jika 1000 km belum dibersihkan: hampir semua nilai akan terhimpit dekat 0).

> Peringatan untuk Pertemuan 10: pada pemodelan, parameter scaling (min, max, mean, std) dihitung **hanya dari data latih**. Menghitungnya dari seluruh data termasuk bentuk *kebocoran data*.

## 4. Agregasi (split–apply–combine)
`groupby` membagi data per kelompok, menghitung ringkasan, lalu menggabungkannya. Hasilnya adalah tabel dengan **unit analisis baru**.

In [ ]:
per_moda = (df.groupby("moda_transportasi")
              .agg(n=("nama", "count"),
                   median_jarak_km=("jarak_km_bersih", "median"),
                   median_waktu_menit=("waktu_tempuh_menit", "median"),
                   rata2_ketertarikan=("tingkat_ketertarikan", "mean"))
              .round(2).reset_index())
per_moda

Unit analisis tabel ini adalah **moda transportasi** (3 baris), bukan mahasiswa. Selalu sertakan `n`: ringkasan untuk kelompok berisi 1 orang bukan "pola".

## 5. Penggabungan Data
### 5a. `merge`: menambah kolom dari tabel lain berdasarkan kunci
Kita punya tabel referensi kecil tentang moda transportasi (didefinisikan sendiri untuk keperluan analisis).

In [ ]:
ref_moda = pd.DataFrame({
    "moda_transportasi": ["Jalan kaki", "Sepeda motor", "Ojek online", "Mobil"],
    "bermotor": [False, True, True, True],
    "kecepatan_wajar_maks_kmjam": [8, 80, 80, 80],
})
ref_moda

In [ ]:
gabung = df.merge(ref_moda, on="moda_transportasi", how="left", validate="many_to_one")
print("Baris sebelum:", len(df), "| sesudah:", len(gabung))
print("Baris tanpa pasangan di tabel referensi:", gabung["bermotor"].isna().sum())
gabung[["nama", "moda_transportasi", "bermotor", "kecepatan_wajar_maks_kmjam"]].head()

Tiga pemeriksaan wajib setelah `merge`:
1. **Jumlah baris** — bertambah? berkurang? (left join seharusnya mempertahankan jumlah baris tabel kiri)
2. **Kunci tanpa pasangan** — muncul sebagai NaN
3. **Kardinalitas** — `validate="many_to_one"` membuat pandas menolak jika kunci di tabel kanan ternyata ganda

| Jenis join | Baris yang dipertahankan |
|---|---|
| `inner` | hanya kunci yang ada di **kedua** tabel |
| `left` | semua baris tabel **kiri** |
| `right` | semua baris tabel **kanan** |
| `outer` | semua baris dari kedua tabel |

In [ ]:
for cara in ["inner", "left", "right", "outer"]:
    print(f"{cara:<6} -> {len(df.merge(ref_moda, on='moda_transportasi', how=cara))} baris")

`right`/`outer` menghasilkan 46 baris karena "Mobil" ada di tabel referensi tetapi tidak ada respondennya.

### Jebakan klasik: kunci ganda menggandakan baris

In [ ]:
ref_salah = pd.concat([ref_moda, pd.DataFrame({"moda_transportasi": ["Sepeda motor"], "bermotor": [True], "kecepatan_wajar_maks_kmjam": [60]})])
salah = df.merge(ref_salah, on="moda_transportasi", how="left")
print("Baris sebelum:", len(df), "| sesudah merge dengan kunci ganda:", len(salah), "<- responden bermotor terhitung dua kali!")
try:
    df.merge(ref_salah, on="moda_transportasi", how="left", validate="many_to_one")
except Exception as e:
    print("Dengan validate='many_to_one' pandas menolak:", type(e).__name__)

### 5b. `concat`: menumpuk baris dari tabel berstruktur sama
Misalnya data Semester 3 dan Semester 5 datang dari dua file terpisah.

In [ ]:
sem3 = df[df["semester"] == "Semester 3"]
sem5 = df[df["semester"] == "Semester 5"]
tumpuk = pd.concat([sem3, sem5], ignore_index=True)
print(len(sem3), "+", len(sem5), "=", len(tumpuk))
df = gabung            # lanjutkan dengan tabel hasil merge
catat("merge (left)", "moda_transportasi + ref_moda", "bermotor, kecepatan_wajar_maks_kmjam", "Menambah atribut moda; many-to-one; jumlah baris tetap 45")

## 6. Pengantar Feature Engineering
*Feature engineering* = membuat variabel baru yang lebih bermakna bagi pertanyaan analitik.

In [ ]:
# (a) rasio dua kolom: kecepatan implisit
df["kecepatan_kmjam"] = (df["jarak_km_bersih"] / (df["waktu_tempuh_menit"] / 60)).round(1)
catat("rasio", "jarak_km_bersih, waktu_tempuh_menit", "kecepatan_kmjam", "Memeriksa konsistensi jarak-waktu per moda")

# (b) penanda (flag) dari aturan
df["kecepatan_tidak_wajar"] = df["kecepatan_kmjam"] > df["kecepatan_wajar_maks_kmjam"]
catat("flag", "kecepatan_kmjam, kecepatan_wajar_maks_kmjam", "kecepatan_tidak_wajar", "Menandai isian jarak/waktu yang perlu dicek")

# (c) pengelompokan (binning) angka menjadi kategori
df["kelompok_waktu_tempuh"] = pd.cut(df["waktu_tempuh_menit"], bins=[0, 5, 10, 60], labels=["<=5 menit", "6-10 menit", ">10 menit"])
catat("binning", "waktu_tempuh_menit", "kelompok_waktu_tempuh", "Memudahkan perbandingan kelompok; batas 5 dan 10 menit")

# (d) penggabungan kategori kecil
df["platform_ringkas"] = np.where(df["platform_belajar"] == "ChatGPT atau AI lainnya", "AI", "Non-AI")
catat("gabung kategori", "platform_belajar", "platform_ringkas", "5 kategori non-AI masing-masing <= 4 responden")

df[["nama", "moda_transportasi", "jarak_km_bersih", "waktu_tempuh_menit", "kecepatan_kmjam", "kecepatan_tidak_wajar", "kelompok_waktu_tempuh", "platform_ringkas"]].head(8)

In [ ]:
print(df["kelompok_waktu_tempuh"].value_counts().sort_index().to_string(), "\n")
print(df["platform_ringkas"].value_counts().to_string(), "\n")
print("Kecepatan tidak wajar:", int(df["kecepatan_tidak_wajar"].sum()), "baris")
df.loc[df["kecepatan_tidak_wajar"], ["nama", "moda_transportasi", "jarak_km_bersih", "waktu_tempuh_menit", "kecepatan_kmjam"]]

**Diskusi:** cukup banyak pejalan kaki yang "kecepatannya" di atas 8 km/jam. Ini bukan berarti mereka berlari — jarak dan waktu adalah **perkiraan pribadi yang dibulatkan** ("1 km, 5 menit"). Fitur turunan mewarisi (dan sering memperbesar) ketidakpastian kolom asalnya.

## 7. Dataset Analisis dan Catatan Transformasi

In [ ]:
assert len(df) == n_awal, "Jumlah baris berubah! Periksa merge."
catatan = pd.DataFrame(transformasi)
catatan

In [ ]:
df.to_csv("dataset_analisis_demo_P6.csv", index=False)
catatan.to_csv("catatan_transformasi_demo_P6.csv", index=False)
print("Tersimpan:", df.shape, "-> dataset_analisis_demo_P6.csv + catatan_transformasi_demo_P6.csv")

## Ringkasan
| Kebutuhan | Transformasi | Hal yang harus dijaga |
|---|---|---|
| Bentuk data tidak cocok | `melt` / `pivot` | 1 baris = 1 observasi |
| Kategori berurutan → angka | encoding ordinal | urutan dan peta eksplisit |
| Kategori tanpa urutan → angka | one-hot | jangan ciptakan urutan palsu |
| Skala angka berbeda jauh | min-max / z-score | parameter dari data latih saja |
| Perlu ringkasan kelompok | `groupby().agg()` | unit analisis berubah; sertakan n |
| Menambah kolom dari tabel lain | `merge` | jumlah baris, kunci, kardinalitas |
| Menumpuk baris | `concat` | struktur kolom sama |
| Variabel lebih bermakna | feature engineering | alasan dan asal-usul setiap fitur |

**Berikutnya (Pertemuan 7):** memilih visualisasi yang tepat dan menyusun cerita data.